<a href="https://colab.research.google.com/github/juanzegarra/Alinhador_Labb/blob/main/automatizacao_rep_seqs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install multiprocessing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
pip install bs4

In [2]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.9 MB/s eta 0:00:00


##Automatização filogenia genes

Para automatizar o processo de retirar uma sequência consenso de um dado gene de um grupo, a partir dessa sequência consenso, utilizar o bowtie para descobrir a sequência em um outro genoma não montado (montagem por referência). Ao final as novas sequências montadas são alinhadas juntamente com as já possuídas para montar filogenia.

**Necessário Biopython e multiprocessing**

In [6]:
from Bio import SeqIO
import subprocess
from multiprocessing import Pool
import re
import os
import glob

gene_fasta = "genes.fasta"

ids = {"COII", "BOSS", "COI", "NADH2", "SINA", "SNF", "WEE", "MARF", "16S"}

genes = list(SeqIO.parse(gene_fasta, "fasta"))

for id in ids:
    pattern = re.compile(rf"\b{id}\b", re.IGNORECASE)
    with open(f"{id}.fasta", "w") as output_file:
        for gene in genes:
            if pattern.search(gene.description):
                output_file.write(f">{gene.id}\n{gene.seq}\n")

genomes = {("/home/rhuanmedeiro/job_soap/Dmeri_cleaned_R1.fastq", "/home/rhuanmedeiro/job_soap/Dmeri_cleaned_R2.fastq", "SER"),
           ("/home/rhuanmedeiro/bowtie_cov_sat/SRR26246248_1_cleaned.fastq", "/home/rhuanmedeiro/bowtie_cov_sat/SRR26246248_2_cleaned.fastq", "ANG")}

def create_consensus(id):
    subprocess.run(["muscle", "-in", f"{id}.fasta", "-out", f"{id}_aln.fasta", "-maxiters", "10"], check=True)
    subprocess.run(["cons", "-sequence", f"{id}_aln.fasta", "-outseq", f"{id}_cons.fasta", "-name", f"{id}_cons"], check=True)
    subprocess.run(["bowtie2-build", f"{id}_cons.fasta", f"{id}_cons"], check=True)

    for genome in genomes:
        subprocess.run([
            "bowtie2", "-x", f"{id}_cons",
            "-1", genome[0], "-2", genome[1],
            "-S", f"{id}_{genome[2]}.sam",
            "--no-unal", "-p", "10"
        ], check=True)

        with open(f"{id}_{genome[2]}.bam", "wb") as f:
            subprocess.run(
                ["samtools", "view", "-b", f"{id}_{genome[2]}.sam"],
                stdout=f, check=True
            )

        subprocess.run([
            "samtools", "sort",
            "-o", f"{id}_{genome[2]}.sorted.bam",
            f"{id}_{genome[2]}.bam"
        ], check=True)

        subprocess.run([
            "samtools", "consensus",
            "-f", "FASTA",
            "-o", f"{id}_{genome[2]}.fasta",
            f"{id}_{genome[2]}.sorted.bam"
        ], check=True)

    #clean up temp files
    bam_sam = glob.glob(f"{id}_*am")
    index = glob.glob(f"{id}_*.bt2")
    remove_files = bam_sam + index
    for f in remove_files:
        os.remove(f)

    samtools_out = f"{id}_{genome[2]}.fasta"
    temp_file = f"temp_{id}_{genome[2]}.fasta"
    with open(samtools_out) as fin, open(temp_file, "w") as fout:
        fout.writelines(
            f">{genome[2]}\n" if l.startswith(">") else l
            for l in fin
        )
    os.replace(temp_file, samtools_out)

    with open(f"{id}_combined.fasta", "wb") as out:
        for genome in genomes:
            with open(samtools_out, "rb") as f:
                out.write(f.read())
        with open(f"{id}.fasta", "rb") as f:
            out.write(f.read())

    subprocess.run([
        "muscle",
        "-in", f"{id}_combined.fasta",
        "-out", f"{id}_combined_aln.fasta",
        "-maxiters", "10"
    ], check=True)


if __name__ == "__main__":
    with Pool(processes=5) as p:
        p.map(create_consensus, ids)


'\ngenomes = {("home/rhuamedeiro/job_soap/Dmeri_cleaned_R1.fastq", "home/rhuamedeiro/job_soap/Dmeri_cleaned_R2.fastq", "SER"),\n           ("home/rhuamedeiro/bowtie_cov_sat/SRR26246248_1_cleaned.fastq", "home/rhuamedeiro/bowtie_cov_sat/SRR26246248_2_cleaned.fastq", "ANG")}\n\ndef create_consensus(ids):\n  for id in ids:\n    subprocess.run(["muscle", "-in", f"{id}.fasta", "-out", f"{id}_aln.fasta", "-maxiters", "10"])\n    subprocess.run(["cons", "-i", f"{id}_aln.fasta", "-o", f"{id}_cons.fasta"])\n    subprocess.run(["bowtie2-build", f"{id}_cons.fasta", f"{id}_cons"])\n\n    for genome in genomes:\n      subprocess.run(["bowtie2", "-x", f"{id}_cons", "-1", f"{genome[0]}", "-2", f"{genome[1]}", "-S", f"{id}_{genome[2]}.sam",\n                      "--no-unal", "-p 10"])\n\n      subprocess.run(["samtools", "view", "-b", f"{id}_{genome[2]}.sam", ">", f"{id}_{genome[2]}.bam"])\n\n      subprocess.run(["samtools", "sort", "-o", f"{id}_{genome[2]}.sorted.bam", f"{id}_{genome[2]}.bam"])\n\n

In [ ]:
import re
#plotar gráfico de montagem

input = ""

def parse_file(input):
  with open(input, "r") as fin, open("assembly_stats.tsv", "w") as fout:
    fout.write("min_len\ttotal_len\tn50\n")
    iter = re.findall(r"\n ===.*\nGaps", fin.read())
    for iteration in iter:
      lines = iteration.split("\n")
      for line in lines:
        if line.startswith("==="):
          min_len = re.sub(r"z=[0-9]+", "\1", line)
        elif line.startswith("Total"):
          total_len = re.sub(r"sum = [0-9]+", "\1", line)
        elif line.startswith("N50"):
          n50 = re.sub(r"N50 = [0-9]+", "\1", line)
      fout.write(f"{min_len}\t{total_len}\t{n50}\n")

parse_file(input)


## Parse do resultado de anotação do repeatmasker
Vai ler o resultado .tsv do RepeatMasker e usar o score de todas anotações para decidir a melhor anotação para dado cluster

In [ ]:
from collections import defaultdict
import re

tsv_file = "/content/annotation_SER_contigs.tabular"
def parse_repeatmasker_output(file_path):
  #Record every cluster and its TE score and annotation
    cluster_annotation = defaultdict(list)
    print(f"executing file {file_path}")
    with open(file_path, "r") as f:
        line_counter = 0
      #skips header
        for line in f:
          line_counter += 1
          if line_counter >= 3:
            #handles mal-formatted tsv files
            parts = line.strip().replace("C\t", "").split("\t")
            score, query, te = parts[0], parts[4], parts[9]

            #keeps only the cluster numer, contig number is not needed
            cluster = re.sub(r"^(CL\d+).*", r"\1", query)

            #excludes simple repeats and low complexity regions
            if te not in {"Simple_repeat", "Low_complexity"}:
              cluster_annotation[cluster].append((score, te))

    with open("cluster_annotation_SER.tsv", "w") as output_file:

      output_file.write("Cluster\tAnnotation\tScore\n")

      for cluster in cluster_annotation:
          cluster_sum = defaultdict(float)

          for score, te in cluster_annotation[cluster]:
              cluster_sum[te] += float(score)

          if cluster_sum:  # avoid error if empty
              max_key, max_value = max(cluster_sum.items(), key=lambda x: x[1])
              output_file.write(f"{cluster}\t{max_key}\t{max_value}\n")

parse_repeatmasker_output(tsv_file)






executing file /content/annotation_SER_contigs.tabular


## TAREAN pós análise

Programa que recebe como input o resultado em html do TAREAN e retorna arquivo fasta com todos consensos identificados para anotação posterior com RepeatMasker/Blast. Também retora métricas de proporção de elementos repetitivos totais na forma de uma tabela .csv.

In [ ]:
from Bio import SeqIO
import re
import pandas as pd
import argparse
from collections import defaultdict
from bs4 import BeautifulSoup
import subprocess

def parse_args():
    parser = argparse.ArgumentParser(
        description="Easy post-processing of TAREAN html report")
    parser.add_argument("html_file", help="Path to the input html file")
    parser.add_argument("-o", type=str, default="output", help="Base name for output directory and files")
    parser.add_argument("--g_size", type=int, default=180000000, help="Genome size for estimated copy number calculation (default for average Drosophila genome size)")
    parser.add_argument("--plot", action="store_true", help="Plot clusters by size (number of reads) and annotation")
    parser.add_argument("--repeatM_file", type=str, help="path to custom annotation file")
    parser.add_argument("--custom_anno", action="store_true", help="Enable overwriting of TAREAN annotation for a external annotation file,\n more information about annotation file format on read.me")
    parser.add_argument("--custom_sat", action="store_true", help="Enable use of custom satDNA fasta file that annotates (note that it must be a exact sequence match)")
    parser.add_argument("--fasta_sats", type=str, help="path to custom satDNA file")
    return parser.parse_args()

class TAREAN_post_processing:
    def __init__(self, html_file, output_base, g_size):
        self.html_file = html_file
        self.output_base = output_base
        self.g_size = g_size
        #set for excluding other types of headers
        self.types = {" Putative satellites (high confidence)", " Putative satellites (low confidence)", " Other"}
        self.clusters, self.metrics = self.parse_html(html_file)

    def parse_html(self, html_file):
        metrics = defaultdict(int)
        with open(html_file, "r") as f:
            html_content = f.read()
            soup = BeautifulSoup(html_content, "html.parser")
            number_of_reads = int(re.search(r'Number of input reads: (\d+)', html_content).group(1))
            metrics["Number of input reads"] = number_of_reads
            analyzed_reads = int(re.search(r'Number of analyzed reads: (\d+)', html_content).group(1))
            metrics["Number of analyzed reads"] = analyzed_reads
            clusters = []
            #iterates over each in h2 header that delimitates tables
            for h2 in soup.find_all("h2"):
                if h2.text in self.types:
                    table = h2.find_next("tbody")
                    #excludes header line of the table
                    for row in table.find_all("tr", class_=lambda x: x != "firstline"):
                        cluster = defaultdict(int)
                        #divides row and extracts relevant cells
                        c = row.find("td", class_="firstcolumn")
                        cells = row.find_all("td", class_="cellinside")
                        cluster["Cluster"] = c.text.strip()
                        proportion = cells[2].text.strip()
                        number_reads = cells[3].text.strip()
                        sat_prob = cells[4].text.strip()
                        consensus = cells[6].text.strip()
                        C_index = cells[9].text.strip()
                        P_index = cells[10].text.strip()
                        annotation_cell = cells[15]

                        #extracts highest ranking annotation
                        clean_annotation = annotation_cell.find_all("b")
                        max_annotation = defaultdict(int)
                        annotation = ""
                        for i in clean_annotation:
                            if len(clean_annotation) <= 1:
                                annotation = clean_annotation[0].text.split("%")[1]
                            elif not i.text:
                              annotation = "NA"
                            else:
                                annotation_value = float(i.text.split("%")[0])
                                annotation_name = i.text.split("%")[1]
                                max_annotation[annotation_name] = annotation_value
                        if max_annotation:
                            max_annotation = max(max_annotation.items(), key=lambda x: x[1])
                            if max_annotation[1] >= 1:
                                annotation = max_annotation[0]

                        cluster["Proportion"] = proportion
                        cluster["Number of reads"] = number_reads
                        cluster["SAT prob"] = sat_prob
                        cluster["consensus"] = consensus
                        cluster["C-index"] = C_index
                        cluster["P-index"] = P_index
                        cluster["Annotation"] = annotation
                        cluster["Classification"] = h2.text.strip()
                        clusters.append(cluster)

        return clusters, metrics
    #corrects number of analyzed reads by excluding contamination
    def correct_metrics(self, metrics, clusters):
        contamination = 0
        for cluster in clusters:
            if cluster["Annotation"] == "Contamination":
                contamination += int(cluster["Number of reads"])
        self.metrics["Number of analyzed reads"] = int(metrics["Number of analyzed reads"]) - contamination

    def estimate_copy_number(self, clusters, metrics, g_size):
        for cluster in clusters:
            cluster["Proportion"] = (float(cluster["Proportion"]) / self.metrics["Number of analyzed reads"]) * 100
            consensus_size = len(cluster["consensus"])
            if consensus_size > 0:
                estimated_copy_number = ((g_size * float(cluster["Number of reads"])) / 100) / consensus_size
            else:
                estimated_copy_number = 0
            cluster["Estimated copy number"] = estimated_copy_number

    def write_files(self, clusters, metrics):
        with open(f"{self.output_base}.tsv", "w") as output_file, open(f"{self.output_base}.fasta", "w") as fasta_out:
            output_file.write("Cluster\tProportion\tNumber of reads\tSAT prob\tconsensus\tC-index\tP-index\tAnnotation\tClassification\tEstimated copy number\n")
            for cluster in clusters:
                output_file.write(f"{cluster['Cluster']}\t{cluster['Proportion']}\t{cluster['Number of reads']}\t{cluster['SAT prob']}\t{cluster['consensus']}\t{cluster['C-index']}\t{cluster['P-index']}\t{cluster['Annotation']}\t{cluster['Classification']}\t{cluster['Estimated copy number']}\n")
                if cluster["consensus"] != "":
                  fasta_out.write(f">{cluster['Cluster']}\t{cluster['Classification']}\n{cluster['consensus']}\n")

    def overwrited_annotation(self, repeatM_file, clusters):
        with open(repeatM_file, "r") as infile:
            custom_annotation = defaultdict(str)
            for line in infile:
                parts = line.strip().split("\t")
                if parts[0] == "Cluster":
                    continue
                else:
                    cluster = parts[0]
                    cluster = re.sub(r"^CL0*", r"", cluster)
                    annotation = parts[1]
                    custom_annotation[cluster] = annotation

        for cluster in clusters:
            cluster_tarean = cluster["Cluster"]
            annotation_tarean = cluster["Annotation"]
            cluster_tarean_clean = re.sub(r"^CL0*", r"", cluster_tarean)
            if cluster_tarean_clean in custom_annotation:
                cluster["Annotation"] = custom_annotation[cluster_tarean_clean]

    def sat_annotation(self, clusters, fasta_sats):
        fasta_seqs = SeqIO.parse(fasta_sats, "fasta")

        for cluster in clusters:
            cluster_seq = cluster["consensus"]
            for seq in fasta_seqs:
                if cluster_seq == seq.seq:
                    cluster["Annotation"] = f"{seq.id}"


    def plot_clusters(self):
        subprocess.run(["plot_clusters.R", f"{self.output_base}.tsv", f"plot{self.output_base}.png"])


def main():
    args = parse_args()
    html_file = args.html_file
    output_base = args.o
    g_size = args.g_size
    #sanity checking custom annotation files


    tarean = TAREAN_post_processing(html_file, output_base, g_size)
    tarean.correct_metrics(tarean.metrics, tarean.clusters)
    tarean.estimate_copy_number(tarean.clusters, tarean.metrics, g_size)
    #if custom library is chosen and provided
    if args.custom_anno and args.repeatM_file:
        repeatM_file = args.repeatM_file
        tarean.overwrited_annotation(repeatM_file, tarean.clusters)
    elif args.custom_anno and not args.repeatM_file:
        print("Provide a valid annotation file")

    if args.custom_sat and args.fasta_sats:
        fasta_sats = args.fasta_sats
        tarean.sat_annotation(tarean.clusters, fasta_sats)
    elif args.custom_sat and not args.fasta_sats:
        print("Provide a valid fasta file")

    tarean.write_files(tarean.clusters, tarean.metrics)

    if args.plot:
        tarean.plot_clusters()

main()

In [ ]:
#!/usr/bin/env Rscript

library(dplyr)
library(ggplot2)
library(viridis)

args <- commandArgs(trailingOnly = TRUE)
input_file <- args[1]
output_file <- args[2]

df <- read.table(input_file, header = TRUE, sep = "\t")

df_sum <- df[order(df$Number.of.reads), ]

df_sum$Annotation[df_sum$Annotation == "" | is.na(df_sum$Annotation) | df_sum$Annotation == "unknown"] <- "NA"
df_sum
ann_colors <- viridis(length(unique(df_sum$Annotation)))
names(ann_colors) <- unique(df_sum$Annotation)
ann_colors["NA"] <- "#898989"

p <- ggplot(df_sum, aes(
    x = Cluster,
    y = Number.of.reads,
    fill = Annotation
)) +
  geom_col(position = "identity", width = 0.64) +
  labs(
    title = element_blank(),
    x = "Clusters",
    y = "Cluster size (n° of reads)"
  ) +
  theme(
    panel.background = element_rect(fill = "#F5F5F5"),
    panel.grid = element_blank(),
    axis.text.x = element_blank(),
    axis.ticks = element_blank(),
    axis.text.y = element_text(size = 10, face = "bold", color = "black"),
  ) +
  scale_fill_manual(values = ann_colors)

ggsave(output_file, p, width = 8, height = 6, dpi = 300)


##Criar arquivo de metadata para haplotype network

In [ ]:
from Bio import SeqIO
import re
from collections import defaultdict

fasta_file = "/content/sequence.fasta"

fasta_seqs = SeqIO.parse(fasta_file, "fasta")

population_dict = {"N26" : "A",
                   "N84" : "A",
                   "N77" : "A",
                   "N95" : "A",
                   "SER" : "A",
                   "N26" : "A",
                   "N96" : "A",
                   "N96" : "A",
                   "J07" : "A",
                   "J7" : "A",
                   "N75" : "A",
                   "N77" : "A",
                   "R65" : "A",
                   "CAN1" : "A",
                   "CAN2" : "A",
                   "SEG" : "A",
                   "JAG" : "A",
                   "VIA" : "B",
                   "J26" : "B",
                   "J24" : "B",
                   "J23" : "B",
                   "N79" : "B",
                   "N87" : "B",
                   "N59" : "C",
                   "N61" : "C",
                   "N09" : "C",
                   "N9" : "C",
                   "N10" : "C",
                   "N78" : "C",
                   "N86" : "C",
                   "N24" : "C",
                   "N2" : "C",
                   "N29" : "C",
                   "N29A9" : "C",
                   "N20" : "C",
                   "N22" : "C",
                   "N31" : "C",
                   "N32" : "C",
                   "N45" : "C",
                   "N63" : "C",
                   "N60" : "C",}

location_dict = defaultdict[str]

def create_metadata(fasta_seqs):
   with open("metadata_file_per.tsv", "a") as output_file:
    output_file.write("sequence\tpopulation\tlocation\n")
    for sequence in fasta_seqs:
      if sequence.id == "SER" or sequence.id == "ANG":
        location = sequence.id
      else:
        description = sequence.description
        parse_description = description.split(" ")
        id = parse_description[0]
        location = parse_description[4]
        if "-" in location:
          location = location.split("-")[0]
      if location in population_dict:
        output_file.write(f"{sequence.description}\t{population_dict[location]}\t{location}\n")

create_metadata(fasta_seqs)

Busco filtering

In [ ]:
from Bio import SeqIO

fasta_file = "/content/Galaxy26-[Busco on dataset 1_ Nucleotide sequences - Specific lineage].fasta"
filter_file = "/content/Galaxy28-[Filter on dataset 24].tabular"

def parse_filter_file(filter_file):
  busco_ids = []
  with open (filter_file, "r") as f:
    for line in f:
      busco_id = line.split("\t")[0]
      busco_ids.append(busco_id)
  return busco_ids

def filtering(fasta_file, busco_ids):
  fasta_seqs = SeqIO.parse(fasta_file, "fasta")
  with open("filtered_busco.fasta", "w") as output_file:
    for seq in fasta_seqs:
      busco_id = seq.id.split("_")[0]
      if busco_id in busco_ids:
        output_file.write(f">{seq.id}\n{seq.seq}\n")

busco_ids = parse_filter_file(filter_file)
filtering(fasta_file, busco_ids)


## Deduplication of sequences

Usando para facilitar o processo de rodar NMDS nas repetições com muitas cópias similares

Mantém apenas a localização da cópia, usando o sets para remover duplicatas de local e sequência.

Tendo em vista que a sequência é nesse formato



```
>LOCALIDADE_ClusterContig_N°decopia
AATCATCGAT
```



In [ ]:
from Bio import SeqIO
import re
from collections import defaultdict

def deduplication(fasta_file):
  #Importar como objeto fasta
  fasta_seqs = SeqIO.parse(fasta_file, "fasta")
  #Usar os sets para reduzir duplicatas
  unique_seqs = set()
  copy_number = defaultdict(int)
  for seq in fasta_seqs:
    seq_seq = seq.seq
    seq_location = seq.id.split("_")[0]
    seq_final = f"{seq_seq}_{seq_location}"
    unique_seqs.add(seq_final)
    copy_number[seq_final] += 1
  with open("deduplicated_DMER28.fasta", "w") as output_file:
    for seq_final in unique_seqs:
      print(seq_final)
      seq_seq, seq_location = seq_final.split("_")
      output_file.write(f">{seq_final}\n{seq_seq}\n")
  with open("metadata_file_DMER28.tsv", "w") as output_file:
    output_file.write("Sequence\tCopies\tlocation\tsat\n")
    for seq_final in unique_seqs:
      seq_seq, seq_location = seq_final.split("_")
      output_file.write(f"{seq_final}\t{copy_number[seq_final]}\t{seq_location}\tDMER28\n")

deduplication("/content/totalDMER28.fasta")

GTTCTTCGGAACTA_Ang
GTTCTTTGACTA_SER
GTTCTTTGGAGGAACT_Ang
GTTCTTTGGAACTA_SER
GTTCTTTGGAACTA_Ang
GTTCTTTGGTACTA_SER
GTTCTATGGAACTA_SER
GCTCTTTGGTACTA_SER
GTTCTATGGAACGA_SER


## Hit_clustering
Realiza a função de agrupar hits retornados pelo blast, se esses estiverem com coordenadas sobrepostas, impedindo que ocorra repetição de loci nas análises posteriores.

Possui a opção de realizar uma expansão e agrupamento iterativo por meio das flags

```
-ex (Size to be expanded)
-cy (Expanding cycles)
```




In [ ]:
#!/usr/bin/env python3
import argparse
import os
from collections import defaultdict
from Bio import SeqIO
import subprocess

##fazer argparse
def parse_args():
    parser = argparse.ArgumentParser(
        description="Join overlapping hits and avoid loci repetition."
    )
    parser.add_argument("Query", help="Path to the input fasta file (note that hits will be clustered independt of query sequence)")
    parser.add_argument("Subject", help="Path to DB files used in BLAST")
    parser.add_argument("-ex", type=int, default=0, help="expanding size (both sides), default = 0")
    parser.add_argument("-cy", type=int, default=0, help="expanding cycles, default = 0")
    parser.add_argument("-o", type=str, default="output.fa", help="output file (fasta)")
    parser.add_argument("-s", type=str, default="\t", help="type of separator used in the file, default = tab")

    return parser.parse_args()

##Run BLAST
def blast_run(db_file, query_file, step):
  tsv_file = "blast_results.tsv"
  subprocess.run(["makeblastdb", "-in", db_file, "-dbtype", "nucl"])

  if step == 1: ## Running blast to first ID loci
    try:
      subprocess.run(["blastn", "-query", query_file, "-db", db_file, "-outfmt", "6 qseqid sseqid sstart send evalue", "-out", tsv_file])
    except Exception as e:
      print(f"Error: {e}, first BLAST didn't run accordingly, examine files.")

  if step == 2: ##Running blast to fecth hit coordinates inside expanded and grouped loci
    try:
      step2_out = "hit_coords_in_loci.bed"
      subprocess.run(["blastn", "-query", query_file, "-db", db_file, "-outfmt", "6 sseqid sstart send qseqid", "-out", step2_out])
    except Exception as e:
      print(f"Error: {e}, final BLAST didn't run accordingly, examine files.")

##parsing tsv using file separator (e.g TSV -> \t)
def processing_tsv(tsv_file, sep):
    loci = set()
    hits = []
    groups = defaultdict(list)
    with open(tsv_file,"r") as f:
      for line in f:
          # Assuming format: Query-ID \sep Subject-id \sep Subject Start \sep Subject End \sep anything else
          parts = line.strip().split(f"{sep}")
          if len(parts) < 5:
              print(f"Skipping malformed line: {line.strip()} (expected at least 6 fields)")
              continue
          q = parts[0] # Query-ID
          # parts[1] is 'Subject-id'
          s = parts[1] # Subject scaffold number
          start = int(parts[2])
          end   = int(parts[3])
          e = float(parts[4]) # E-value

          # normalize orientation
          if start > end:
              start, end = end, start

          hits.append((q, s, start, end, e))
          loci.add(s)

      # Re-grouping all hits by subject after initial parsing and normalization
      for q, s, start, end, e in hits:
        groups[s].append((q, start, end, e))

      # Sort hits within each subject by start coordinate
      for s in groups:
        groups[s].sort(key=lambda x: x[1])
    return groups

def overlaps(a_start, a_end, b_start, b_end):
  if b_start <= a_end and b_end >= a_start:
    return True
  else:
    return False

def compare_ranges(groups):
  clustered_groups = defaultdict(list)
  for s in groups:
    if not groups[s]: # Skip if no hits for this subject
        continue

    # Initialize with the first hit for the current subject
    current_q, current_start, current_end, current_e = groups[s][0]

    for i in range(1, len(groups[s])):
        next_q, next_start, next_end, next_e = groups[s][i]
        # Check for overlap
        if overlaps(current_start, current_end, next_start, next_end) or overlaps(next_start, next_end, current_start, current_end):
            # Merge overlapping ranges: take min of starts, max of ends.
            # Keep q and e from the first hit that initiated the merged block.
            current_start = min(current_start, next_start)
            current_end = max(current_end, next_end)

        else:
            # If no overlap, add the current merged range to the clustered_groups
            clustered_groups[s].append((current_q, current_start, current_end, current_e))
            # Start a new merged range with the non-overlapping hit
            current_q, current_start, current_end, current_e = next_q, next_start, next_end, next_e

    # Add the last (or only) processed range for the current subject
    clustered_groups[s].append((current_q, current_start, current_end, current_e))

  return clustered_groups

def expand_ranges(groups, expanding_size):
  expanded_groups = defaultdict(list)
  for s in groups:
    for q, start, end, e in groups[s]:
      expanded_start = max(0, start - expanding_size)
      expanded_end = end + expanding_size
      # Preserve all original hit information along with expanded coordinates
      expanded_groups[s].append((q, expanded_start, expanded_end, e))
  return expanded_groups

def write_tsv(clustered_groups):
  with open("clustered_ranges.bed", "w") as f:
    # Iterate through each subject (s) and its list of clustered ranges
    for s, ranges_list in clustered_groups.items():
      # Iterate through each clustered range (q, start, end, e) for the current subject
      for q, start, end, e in ranges_list:
          f.write(f"{s}\t{start}\t{end}\t{q}\n")


def get_fasta(grouped_tsv_file, subject, out_file):
  subprocess.run(["bedtools", "getfasta", "-fi", subject, "-bed", grouped_tsv_file, "-fo", out_file])
  return out_file

def main():
  #parsing argv
  args = parse_args()
  query_file = args.Query
  db_file = args.Subject
  blast_run(db_file, query_file, 1)
  tsv_file = "blast_results.tsv"
  expanding_cycles = args.cy
  expanding_size = args.ex
  out_file = args.o
  sep = args.s

  # Initial processing of the TSV file
  groups = processing_tsv(tsv_file, sep)
  groups = compare_ranges(groups)

  # Perform iterative expansion and clustering if specified

  while expanding_cycles > 0:
    expanding_cycles -= 1
    expanded_groups = expand_ranges(groups, expanding_size)
    groups = compare_ranges(expanded_groups) # Update 'groups' with the clustered results for the next iteration

  # Write the final clustered ranges to a BED file

  write_tsv(groups)
  get_fasta("clustered_ranges.bed", db_file, out_file)
  blast_run(out_file, query_file, 2)


if __name__ == "__main__":
  main()

IndentationError: expected an indented block after 'if' statement on line 109 (ipython-input-2129668270.py, line 112)

## mob_counter

Versão final, com implementação do seaborn e matplotlib ao invés do script no R.
Código mais limpo e sem excessos de if-else

In [ ]:
#!/usr/bin/env python3
import argparse
import os
from collections import defaultdict
from Bio import SeqIO
import re
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Mapping between Wicker (2007) nomenclature and families
Wicker_table = {
    "DTT": "TcMar",
    "DTA": "hAT",
    "DTM": "Mutator",
    "DTE": "Merlin",
    "DTR": "CMC",
    "DTP": "P",
    "DTB": "PiggyBac",
    "DTH": "PIF-Harbinger",
    "DTC": "CMC",
    "unknown": "Unknown"
}
#Complete name list
normal = ["LINE", "SINE", "LTR", "PLE", "DIRS", "Unknown", "Sola", "Tc-Mar", "Helitron", "Mutator", "CMC-EnSpm",
          "CACTA", "Merlin", "P", "MULE", "CMC-Transib", "Maverick", "PIF-Harbinger", "PiggyBac"]

#Parsing arguments
def parse_args():
    parser = argparse.ArgumentParser(
        description="Count transposon families based on FASTA headers produced by EDTA."
    )
    parser.add_argument("fasta_file", help="Path to the input FASTA file")
    parser.add_argument(
        "-N",
        action="store_true",
        help="Use full nomenclature (e.g. DNA/hAT, LINE, etc.)",
    )
    parser.add_argument(
        "-W",
        action="store_true",
        help="Use Wicker (2007) nomenclature (e.g. DTT -> TcMar, DTA -> hAT, etc.)",
    )
    return parser.parse_args()


def mob_counter(fasta_file, N, W):
  #Sanity check
    if (N and W) or (not N and not W):
        raise ValueError("You must choose exactly one mode: -N OR -W")

    print(f"Processing {fasta_file}...")
    mob_counts = defaultdict(int)

    #Parsing an entry of type ">seq_id#transposon_type"
    for record in SeqIO.parse(fasta_file, "fasta"):
        family_fields = record.id.split("#")
        if len(family_fields) < 2:
            print(f"Skipping malformed header: {record.id}")
            continue

        correct_family = family_fields[1]

        # ===== FULL NOMENCLATURE =====
        if N and not W:
            match = next((te for te in normal if te in correct_family), None)
            if match:
                family_name = match
            else:
                print(f"Unrecognized Mob full nomenclature: {correct_family}")
                continue
            mob_counts[family_name] += 1

        # ===== WICKER NOMENCLATURE =====
        elif W and not N:
            if correct_family.startswith(("DNA/", "MITE/")):
                clean_name = re.sub(r"^(DNA/|MITE/)", "", correct_family)
                family_name = Wicker_table.get(clean_name, clean_name)
            elif correct_family.startswith("RC/"):
                clean_name = re.sub(r"^RC/", "", correct_family)
                family_name = Wicker_table.get(clean_name, clean_name)
            else:
                match = next((te for te in normal if te in correct_family), None)
                if match:
                    family_name = match
                else:
                    print(f"Unrecognized Wicker family: {correct_family}")
                    continue
            mob_counts[family_name] += 1

    # --- Output TSV ---
    file_base = os.path.splitext(os.path.basename(fasta_file))[0]
    out_tsv = f"transposons_{file_base}.tsv"

    with open(out_tsv, "w") as file:
        file.write("Family\tCopies\n")
        for family, count in mob_counts.items():
            file.write(f"{family}\t{count}\n")

    print(f"Counts written to {out_tsv}")

    # --- Generate plot ---
    try:
        table = pd.read_csv(out_tsv, sep='\t')

        sns.set_theme(style="whitegrid")
        unique_colors = sns.color_palette("Set3", n_colors=len(table))

        plt.figure(figsize=(12, 6))
        bar = sns.barplot(x="Family", y="Copies", data=table, palette=unique_colors, edgecolor="None")

        plt.title("Número de cópias por família de transposons", fontsize=16, fontweight="bold")
        plt.xlabel("Família de transposons", fontsize=14)
        plt.ylabel("Nº de subfamílias", fontsize=14)
        plt.grid(False)

        plt.xticks(rotation=45, ha="right", fontsize=11)
        plt.yticks(fontsize=11)

        plot_filename = f"barplot_{file_base}.png"
        plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
        print(f"Plot saved as {plot_filename}")

    except Exception as e:
        print(f"Error generating plot: {e}")

    print("Finished.")


def main():
    args = parse_args()
    mob_counter(args.fasta_file, args.N, args.W)


if __name__ == "__main__":
    main()

## Instalar o biopython para conseguir executar as células

##Processar arquivo de Scaffolding
(apenas manter o nome do header simples)

In [ ]:
from Bio import SeqIO

genom1 = "/content/standard_output.final.scaffolds.fasta"

genom1 = SeqIO.parse(genom1, "fasta")
with open("genom1.fasta", "w") as output_file:
  for seq1 in genom1:
    processed = seq1.id.split("|")[0]
    fa = seq1.seq
    output_file.write(f">{processed}\n{fa}\n")

##Diferenciar arquivos fasta
(Por meio de conjuntos)

In [ ]:
from Bio import SeqIO

db1 = "/content/RepBase-nov25.fasta"
db2 = "/content/drosophila_repbase2025.fa"

db1 = SeqIO.parse(db1, "fasta")
db2 = SeqIO.parse(db2, "fasta")
db1_ids = set()
db2_ids = set()
for seq1 in db1:
  db1_ids.add(seq1.id)
for seq2 in db2:
  db2_ids.add(seq2.id)
difference = db1_ids.difference(db2_ids)
print(len(difference))
for seq in difference:
  if "(" not in seq:
    print(seq)

89
Buster_DTan
Buster_DBoc
Buster_DAn
SSU-rRNA_Hsa
PENELOPE
LSU-rRNA_Cel
LSU-rRNA_Hsa
Buster_DAzt
SSU-rRNA_Dme
Buster_DNik
LSU-rRNA_Dme
SSU-rRNA_Cel
MARINA
Buster_DroWat
Buster_DAsa
Buster_DAth
Buster_DAur
Buster_DBir
Buster_DTru
Buster_DSub


In [ ]:
import re
from Bio import SeqIO
import sys

if len(sys.argv) > 1:
  print(f"Arguments: {sys.argv[1:]}")
else:
  print("No command-line arguments provided.")
fasta_f = sys.argv[1]

def x(fa):
    seq = SeqIO.parse(fa, "fasta")
    j = []
    i = []
    for s in seq:
      if "J" in s.id:
        j.append(s.seq)
        i.append(s.id)
      else:
        continue
    value = j.index(max(j))
    print(i[value])
    print(max(j))
x(fasta_f)


In [ ]:
from Bio import SeqIO
import re

transpo_file = "/content/repeatmodeler_DMER_Kim.fasta"

def filter_transpos(fasta_file):
  with open(fasta_file, "r") as file:
    fasta_data = SeqIO.parse(file, "fasta")
    matches = []
    for seq in fasta_data:
      match = re.search(r"Unknown|unknown", seq.id)
      if match:
        matches.append(seq)
        with open("Unknown_transposons_ANG.fasta", "a") as output_file:
          SeqIO.write(seq, output_file, "fasta")
    print(matches)
    if len(matches)<=0:
      print("No matches found")
filter_transpos(transpo_file)

[SeqRecord(seq=Seq('TTAATAGATTGGGACTGACTTAACGGAATTTAATGAAACTTTCAGCATACGATT...NTT'), id='rnd-1_family-31#Unknown', name='rnd-1_family-31#Unknown', description='rnd-1_family-31#Unknown ( RepeatScout Family Size = 141, Final Multiple Alignment Size = 100, Localized to 33 out of 34 contigs )', dbxrefs=[]), SeqRecord(seq=Seq('AAACAAGTAAGAGTGCTCTAGTCGAGACTGCTCGACTAGGAGATACCCTGAGCC...ATT'), id='rnd-1_family-8#Unknown', name='rnd-1_family-8#Unknown', description='rnd-1_family-8#Unknown ( RepeatScout Family Size = 677, Final Multiple Alignment Size = 100, Localized to 33 out of 34 contigs )', dbxrefs=[]), SeqRecord(seq=Seq('ATGTTCAGGCATGAACATAGAAATATATATAAATATGTAAACATGCCTTCTACG...TCA'), id='rnd-1_family-41#Unknown', name='rnd-1_family-41#Unknown', description='rnd-1_family-41#Unknown ( RepeatScout Family Size = 100, Final Multiple Alignment Size = 100, Localized to 33 out of 34 contigs )', dbxrefs=[]), SeqRecord(seq=Seq('ATGTATATATATATATATATATATATATATATATATATATATATATATATATAT...TCT'), id='rnd-1_

###Blast e Dotplot automatizado (YASS)

In [ ]:
import subprocess
import os
import re
from Bio import SeqIO
import argparse

def parse_args():
    parser = argparse.ArgumentParser(
        description="Plot YASS dotplot based on an query sequence of interest"
    )

    # Positional (required) arguments
    parser.add_argument("query", help="Query FASTA file")
    parser.add_argument("genome", help="Genome FASTA file")
    parser.add_argument("yfile", help="Subject file for YASS")
    parser.add_argument("output", help="Output file name")
    args = parser.parse_args()
    return args

class Query:
   def __init__(self, genome, record):
    self.genome = genome
    self.record = record
    self.yfile = None
    self.tsv = f"{self.seq}."
    self.fasta_file = f"{self.record}_YASS.fa"

    def get_fasta(self, bed_file, fasta_file):
      try:
          subprocess.run(
              [
                  "bedtools",
                  "getfasta",
                  "-fi",
                  self.genome_file,
                  "-bed",
                  bed_file,
                  "-fo",
                  fasta_file,
              ]
          )
      except subprocess.CalledProcessError as e:
          print(f"Error: {e}")
          sys.exit(1)


    def blast_yass(self, self.record, self.genome, self.tsv):
        work_d = os.getcwd()
        genome_prefix = os.path.splitext(self.genome)[0]


        print(f"Running BLAST for {self.query}...")

        subprocess.run([
                "blastn",
                "-query",
                query,
                "-db",
                ,
                "-outfmt",
                "6 qseqid sseqid sstart send",
                "-out",
                output,
           ])
    def yass_prep():
       with open(self.results_tsv, "r") as fin, open(
            f"{self.record}_YASS.bed", "w"
        ) as fout:
            for line in fin:
                qseqid, sseqid, sstart, send = line.strip().split()
                sstart, send = int(sstart), int(send)

                # normalize strand orientation
                start, end = min(sstart, send), max(sstart, send)

                # expand window depending on cycle
                yass_size = len(self.query.seq)
                start = max(0, start - yass_size)  # avoid negative coords
                end = end + yass_size

                # BED requires 0-based start
                fout.write(f"{sseqid}\t{start - 1}\t{end}\n")
        get_fasta(f"{self.record}_YASS.bed", self.fasta_file)
      return self.fasta_file

    def yass_exec(query, Subject):



def main():
    args = parse_args()
    query = args.query
    genome = args.genome
    yfile = args.yfile
    output = args.output

    ##Sanity checking

    if not os.path.isfile(os.path.join(work_d, genome)):
        print("Genome file is missing")
    elif not os.path.isfile(os.path.join(work_d, self.query)):
        print("Query file missing")
    else:
        subprocess.run(
            [
                "makeblastdb",
                "-in",
                self.genome,
                "-dbtype",
                "nucl",
                "-out",
                genome_prefix,
            ]
        )

    query = Query(genome, record)
    query.blast_yass(query, genome, yfile)

if __name__ == "__main__":




IndentationError: unindent does not match any outer indentation level (<tokenize>, line 101)

## Filtrar seqs EnSpm em arquivo de db.fasta

In [ ]:
import re
from Bio import SeqIO
from google.colab import files

fasta_file = "/content/drosophila_repbase2025.fa"
def droso_filter(fasta_file):
  with open(fasta_file, "r") as file:
    fasta_data = SeqIO.parse(file, "fasta")
    matches = []
    for seq in fasta_data:
      hits = re.findall(r"EnSpm-.{1,3}D.{2,4}", seq.id)
      matches.extend(hits)
      if len(matches) > 0:
        with open("Drosophila_EnSpm.fasta", "a") as output_file:
          SeqIO.write(seq, output_file, "fasta")
    print(matches)
    if len(matches)<=0:
      print("No matches found")
droso_filter(fasta_file)


['EnSpm-N3_DEl', 'EnSpm-N3_DTa', 'EnSpm-N5_DSuz', 'EnSpm-N1_DTa', 'EnSpm-N4_DTa', 'EnSpm-N3_DSuz', 'EnSpm-1_DTa', 'EnSpm-N7_DSuz', 'EnSpm-N7_DTa', 'EnSpm-N5_DTa', 'EnSpm-N2_DGr', 'EnSpm-7_DTa', 'EnSpm-N9_DTa', 'EnSpm-8_DTa', 'EnSpm-N9_DEl', 'EnSpm-N4_DSuz', 'EnSpm-5_DTa', 'EnSpm-N1_DMo', 'EnSpm-1_DWil', 'EnSpm-9_DTa', 'EnSpm-N1_DVi', 'EnSpm-N8_DTa', 'EnSpm-N7_DEl', 'EnSpm-3_DWil', 'EnSpm-6_DTa', 'EnSpm-N6_DTa', 'EnSpm-N8_DSuz', 'EnSpm-2_DWil', 'EnSpm-N1_DGr', 'EnSpm-N2_DSuz', 'EnSpm-2_DSuz', 'EnSpm-N8_DEl', 'EnSpm-N1_DEl', 'EnSpm-N6_DEl', 'EnSpm-3_DSuz', 'EnSpm-N4_DEl', 'EnSpm-N3_DAlb', 'EnSpm-10_DTa', 'EnSpm-N9_DSuz', 'EnSpm-5_DWil', 'EnSpm-N2_DEl', 'EnSpm-N5_DEl', 'EnSpm-4_DWil', 'EnSpm-N2_DAlb', 'EnSpm-1_DSuz', 'EnSpm-1_DEu', 'EnSpm-1_DAlb', 'EnSpm-4_DTa', 'EnSpm-2_DTa', 'EnSpm-N6_DSuz', 'EnSpm-N2_DTa', 'EnSpm-4_DSuz', 'EnSpm-3_DTa']


In [ ]:
import re
from Bio import SeqIO
from google.colab import files

fasta_file = "/drosophila_repbase2025.fa"
def header_clean(fasta_file):
  with open(fasta_file, "r") as file, open("drosophila_repbase2025_correct.fa", "w") as outfile:
    for line in file:
      if line.startswith(">"):
              # Remove leading '>'
              header = line[1:].strip()
              # Replace tabs or multiple spaces with a single underscore
              header = re.sub(r"[\t ]+", "_", header)
              # Keep only the first token before any space/underscore combo (optional)
              # header = header.split()[0]
              print(header)
              outfile.write(f">{header}\n")
      else:
            # Remove tabs from sequence lines (if any)
            seq = line.strip().replace("\t", "").upper()
            outfile.write(seq + "\n")
header_clean(fasta_file)


BEL-5_DMel-LTR_BEL_Drosophila_melanogaster
DM412_LTR_Gypsy_Drosophila_melanogaster
Gypsy-11_DMel-I_Gypsy_Drosophila_melanogaster
Gypsy-31_DMel-I_Gypsy_Drosophila_melanogaster
Copia-2B_DMel-I_Copia_Drosophila_melanogaster
M4DM_Transib_Drosophila_melanogaster
Gypsy-28_DMel-LTR_Gypsy_Drosophila_melanogaster
DMR_DV_Transposable_Element_Drosophila_virilis
COPIA_DM_I_Copia_Drosophila_melanogaster
Gypsy-7_DMel-I_Gypsy_Drosophila_melanogaster
DMHMR2_DMHMR2_Drosophila_melanogaster
Gypsy-30_DMel-I_Gypsy_Drosophila_melanogaster
MDG3_DM_Gypsy_Drosophila_melanogaster
INVADER4_I_Gypsy_Drosophila_melanogaster
DM412_I_Gypsy_Drosophila_melanogaster
BS3_DM_Jockey_Drosophila_melanogaster
GYPSY2_LTR_Gypsy_Drosophila_melanogaster
DMSAT6_SAT_Drosophila_melanogaster
STALKER4_LTR_Gypsy_Drosophila_melanogaster
Spoink_LTR_Gypsy_Drosophila_melanogaster
DMRT1A_R1_Drosophila_melanogaster
Copia1-I_DM_Copia_Drosophila_melanogaster
GYPSY10_LTR_Gypsy_Drosophila_melanogaster
STALKER2_LTR_Gypsy_Drosophila_melanogaster
S

## Importar arquivo multifasta das sequências usando o import do google colab



In [ ]:
from google.colab import files

uploaded = files.upload()


KeyboardInterrupt: 

# **A função seqheader vai organizar os cabeçalhos das sequências de fasta.**

Tenha as sequências por exemplo:
```
>CL3Contig139
CTTCAT
CTTCAT
CTTCAT
>CL3Contig140
CTTCAT
CTTCAT
CTTCAT
```
Após a execução da função, as sequências se tornarão:
```
>CL3Contig139_copy_1
CTTCAT
>CL3Contig139_copy_2
CTTCAT
>CL3Contig139_copy_3
CTTCAT
>CL3Contig140_copy_1
CTTCAT
>CL3Contig140_copy_2
CTTCAT
>CL3Contig140_copy_3
CTTCAT
```
Importante notar que essas sequências mantém a nomenclatura antiga



In [ ]:
def seqheader(file, newfile):
    fixedlines = []
    current_header = None
    copy_number = 1
    first_sequence = False

    with open(file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                current_header = line
                copy_number = 1
                first_sequence = True
                fixedlines.append(f"{current_header}_copy_{copy_number}")

            else:
                if not first_sequence:
                    copy_number += 1
                    fixedlines.append(f"{current_header}_copy_{copy_number}")

                fixedlines.append(line)
                first_sequence = False

    with open(newfile, 'w') as f_out:
        for item in fixedlines:
            f_out.write(f"{item}\n")

seqheader("CL1eCL3.fasta", "header_CL1eCL3.fasta")


## reverse_seqs
A função reverse_seqs transforma sequências que estão na forma 3'5' em 5'3'

Importante notar que essas sequências já devem ser previamente conhecidas, ou seja, seu sentido deve ter sido anotado durante a mineração

Antes da execução

```
>CL3Contig2_copy_1
ATGAAG
>CL3Contig2_copy_2
ATGAAG
>CL3Contig2_copy_3
ATGAAG
```
Após a execução da função, as sequências se tornarão:
```
>CL3Contig2_copy_1_reversed
CTTCAT
>CL3Contig2_copy_2_reversed
CTTCAT
>CL3Contig2_copy_3_reversed
CTTCAT
```
Importante notar que demais anotações no cabeçalho da sequência como "(reversed)" do Geneious ou "HOR" continuam intactas

In [ ]:
from Bio import SeqIO

def reverse_seqs(input_fasta, output_fasta):
    rev_seqs = [
        'ATAAAG', 'ATTAAT', 'ATAAAT', 'ATGAAG', 'AGAAAG', 'AGGAAG', 'TAGAAG',
        'AAGAAG', 'TTGAAG', 'ATGTAG', 'ATGAAA', 'GTGAAG', 'ATGAGG', 'ATGATG',
        'ATAATG', 'ATCAAG', 'ATCAT', 'ATACAG', 'ATGGAG', 'ATGAAT', 'ACGAAG',
        'AGGTAG', 'CTGAAG', 'ATTAAG', 'ATTTAT'
    ]
    rev_seqs = [seq.upper() for seq in rev_seqs]


    with open(output_fasta, "w") as out_handle:
        for record in SeqIO.parse(input_fasta, "fasta"):
            seq_original = str(record.seq).upper()
            if seq_original in rev_seqs:
                record.seq = record.seq.reverse_complement()
                record.id += "_reversed"
                record.description = ""
            SeqIO.write(record, out_handle, "fasta")


reverse_seqs("header_CL1eCL3.fasta", "header_CL1eCL3_modified.fasta")


## repeat_counter

A função repeat_counter vai contar quantas vezes cada sequência idêntica se repete durante o arquivo de fasta, no final é retornada sequência e seu número de repetições



In [ ]:
from collections import defaultdict
from Bio import SeqIO
import os

def repeat_counter(fasta_file):

  repeats = defaultdict(int)

  for record in SeqIO.parse(fasta_file, "fasta"):
    sequence = str(record.seq)
    repeats[sequence] += 1
  for seq, count in repeats.items():
      print(f"{seq}\t{count}")

repeat_counter("header_DMER28_rev.fasta.fasta")

In [ ]:
#!/usr/bin/env python3
from collections import defaultdict
from Bio import SeqIO

def mob_counter(fasta_file):
  mob_counter = defaultdict(int)
  for record in SeqIO.parse(fasta_file, "fasta"):
    sequence = str(record.id)
    family = sequence.split("#")
    mob_counter[family[1]] += 1
  file_name = os.path.basename(fasta_file)
  file_name = file_name.replace(".fasta", "")
  open(f"transposons_{file_name}.tsv", "w") as file:
    for seq, count in mob_counter.items():
      file.write(f"{seq}\t{count}\n")


mob_counter(fasta_file)

### Mob_Counter
Mob_counter é um programa para extrair o número de instâncias de famílias de Transposons em um arquivo fasta (incluindo outputs do RepeatModeler2) automaticamente plotando um gráfico de barras no R com o número de famílias pertecente a cada família de transoposon no genoma

In [ ]:
#!/usr/bin/env python3
import argparse
import os
import subprocess
from collections import defaultdict
from Bio import SeqIO
import re

Wicker_table = {"DTT" : "TcMar", "DTA" : "hAT", "DTM": "Mutator", "DTE" : "Merlin", "DTR" : "CMC-Transib",
                        "DTP" : "P", "DTB":"PiggyBac", "DTH":"PIF-Harbinger", "DTC":"CMC-Transib", "TcMar-Tc1" : "TcMar",
                        "TcMar-Mariner" : "TcMar", "Helitron":"Helitron"}
type_1 = ["LINE", "SINE", "LTR", "PLE", "DIRS"]
def parse_args():
    parser = argparse.ArgumentParser(description="Plot bar-graph based on ocurrence of family of transposons in a .fa file")
    parser.add_argument("fasta_file", help="Path to the input FASTA file")
    parser.add_argument( "-N", action="store_true", help="Nomenclature used - N or W (Whole name or three letter based, see Wicker, 2007)")
    parser.add_argument( "-W", action="store_true", help="Wicker nomenclature")
    args = parser.parse_args()

    return parser.parse_args()

def mob_counter(fasta_file):
        print(f"Processing {fasta_file}...")
        mob_counter = defaultdict(int)
        for record in SeqIO.parse(fasta_file, "fasta"):
            sequence = str(record.id)
            family = sequence.split("#")
            correct_family = family[1]
        #Selecting type of nomenclature
            if (N == True) and (W == False):
                print("Full nomenclature used")

                if correct_family.startswith("DNA/") == True:
                    if correct_family.startswith("DNA/hAT") == True:
                        correct_family = correct_family.replace(f"{correct_family}", "hAT")
                        mob_counter[family[1]] += 1
                    else:
                        correct_family = correct_family.replace("DNA/", "")
                        mob_counter[family[1]] += 1
                if correct_family.startswith("PLE/") == True:
                    correct_family = "PLE"
                    mob_counter[family[1]] += 1
                elif correct_family.startswith("LTR/") == True:
                    correct_family = "LTR"
                    mob_counter[family[1]] += 1
                elif correct_family.startswith("LINE/") == True:
                    correct_family = "LINE"
                    mob_counter[family[1]] += 1
                elif correct_family.startswith("SINE/") == True:
                    correct_family = "SINE"
                    mob_counter[family[1]] += 1
                elif correct_family.startswith("RC/Helitron") == True:
                    correct_family = "Helitron"
                    mob_counter[family[1]] += 1
                else:
                    print(f"{correct_family} has a non parsable header")

            elif (N == False) and (W == True):
              print("Wicker nomenclature used")

              if correct_family.startswith("DNA/") == True or (correct_family.startswith("MITE/") == True):
                  correct_family = re.sub(r"DNA/|MITE/", "", correct_family)
                  correct_family = Wicker_table[correct_family]
                  mob_counter[family[1]] += 1
              elif correct_family.startswith("RC/") == True:
                  correct_family = correct_family.replace("RC/", "")
                  mob_counter[family[1]] += 1
              else:
                for te in type_1:
                  match = re.search(te, correct_family)
                  if match:
                    correct_family = te
                    mob_counter[family[1]] += 1
                    break
            else:
              print("Wrong use of nomenclature parameters")

            file_name = os.path.basename(fasta_file).replace(".fasta", "")
            input_name = f"transposons_{file_name}.tsv"
            out_name = f"transposons_{file_name}.png"

        with open(input_name, "w") as file:
            file.write("Family\tCopies\n")
            for seq, count in mob_counter.items():
                file.write(f"{seq}\t{count}\n")
        subprocess.run([ "mob_counter_plot.R", input_name, out_name],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)


def main():
    args = parse_args()
    fasta_file = args.fasta_file
    N = args.N
    W = args.W
    mob_counter(fasta_file)


if __name__ == "__main__":
    main()


In [ ]:
from google.colab import files

files.download('header_DMER28sequencias.fasta')

In [ ]:
from collections import defaultdict
from Bio import SeqIO

def reverse_complement(fasta_file):

  reverse_complement_sequences = []

  for record in SeqIO.parse(fasta_file, "fasta"):
    sequence = str(record.seq)
    reverse_complement_sequence = str(record.seq.reverse_complement())
    reverse_complement_sequences.append(f">{record.id}(reversed)")
    reverse_complement_sequences.append(reverse_complement_sequence)
  open("DMER28_reversed_sequences", "w").write("\n".join(reverse_complement_sequences))

reverse_complement("header_DMER28_rev.fasta.fasta")